# QLoRA: 4-bit NormalFloat NF4 Quantization

This notebook maps floating point values to the optimal non-uniform NormalFloat 16-bin centroids array, simulating QLoRA parameter savings.

In [1]:
import torch
import numpy as np

# Simple NF4 quantization centroids lookup table (representing 16 bins for N(0,1))
NF4_CENTROIDS = torch.tensor([
    -1.0, -0.6961917, -0.51508895, -0.3854331, -0.2759082, -0.1727768, -0.06997282, 0.0,
    0.07658493, 0.17702854, 0.27961054, 0.3862211, 0.5022872, 0.64010215, 0.8354714, 1.0
])

def quantize_nf4(weight_tensor):
    # Normalize tensor to range [-1, 1]
    max_val = torch.max(torch.abs(weight_tensor))
    scaled_weight = weight_tensor / max_val
    
    # Map to nearest centroid index
    diff = torch.abs(scaled_weight.unsqueeze(-1) - NF4_CENTROIDS)
    quant_indices = torch.argmin(diff, dim=-1)
    return quant_indices, max_val

def dequantize_nf4(quant_indices, max_val):
    dequant_weight = NF4_CENTROIDS[quant_indices] * max_val
    return dequant_weight

# Create toy weight matrix
weights = torch.randn(4, 4)
indices, scale = quantize_nf4(weights)
dequant = dequantize_nf4(indices, scale)

print("Original Weights:\n", weights)
print("\nQuantized Centroid Indices:\n", indices)
print("\nDequantized Reconstruction:\n", dequant)
print(f"\nMax reconstruction absolute error: {torch.max(torch.abs(weights - dequant)).item():.4f}")


Original Weights:
 tensor([[-0.8956,  3.4529, -1.2503,  0.1914],
        [-0.8687, -0.5461, -1.1107, -2.4373],
        [-1.4182, -0.5517, -0.6171,  0.5051],
        [ 0.1018,  1.2995,  1.4050, -1.8027]])

Quantized Centroid Indices:
 tensor([[ 4, 15,  3,  8],
        [ 4,  5,  4,  1],
        [ 3,  5,  5,  9],
        [ 7, 11, 11,  2]])

Dequantized Reconstruction:
 tensor([[-0.9527,  3.4529, -1.3309,  0.2644],
        [-0.9527, -0.5966, -0.9527, -2.4039],
        [-1.3309, -0.5966, -0.5966,  0.6113],
        [ 0.0000,  1.3336,  1.3336, -1.7786]])

Max reconstruction absolute error: 0.1580


### Output Explanation
The output shows the original FP32 tensor, the mapped 4-bit indices (values 0–15), and the dequantized reconstruction. The reconstruction error remains bounded, verifying NF4 efficiency.